# Data leakage

**Data leakage** is one of the most critical methodological errors in machine learning. It occurs whenever information from outside the training dataset is used to create the model. This includes not only the fitting of model parameters (weights), but also preprocessing decisions, hyperparameter tuning, and algorithm selection.

The consequence of data leakage is always the same: the reported performance metrics become **optimistically biased**, meaning they overestimate how well the model will perform on truly unseen data. A model affected by leakage may look excellent during development but fail dramatically in production.

Data leakage can be classified into three main types.

## Feature leakage (preprocessing contamination)

This occurs when statistics computed from the entire dataset (including test data) are used to transform features during preprocessing. The model indirectly "sees" properties of the test set through these statistics.

**Example**: Computing the mean and standard deviation for standardization (Z-score normalization) using the entire dataset, then applying it to both train and test sets.

```python
# WRONG: Leakage through preprocessing
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaler.fit(X)  # Fitted on ALL data, including test!
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

# CORRECT: Fit only on training data
scaler = StandardScaler()
scaler.fit(X_train)  # Fitted ONLY on training data
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)
```

This is why scikit-learn [pipelines](pipelines.ipynb) are essential: they ensure that all preprocessing steps are fitted only on training data, also inside cross-validation.

Let's see the two versions on the California housing data. The difference in the learned mean is tiny, which is exactly why this kind of leakage goes unnoticed: it rarely breaks anything, it just makes every reported score slightly optimistic.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

housing = pd.read_csv("../data/housing.csv")
X = housing.drop(columns="median_house_value")
y = housing["median_house_value"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

leaky_scaler = StandardScaler().fit(X[["median_income"]])  # WRONG: the test rows influence the statistics
clean_scaler = StandardScaler().fit(X_train[["median_income"]])  # fitted on training rows only

print(f"mean learned with leakage: {leaky_scaler.mean_[0]:.4f}")
print(f"mean learned correctly:    {clean_scaler.mean_[0]:.4f}")

mean learned with leakage: 3.8707
mean learned correctly:    3.8808


## Target leakage

This occurs when a feature contains information that is derived from (or strongly correlated with) the target variable in a way that would not be available at prediction time in production.

**Example**: Predicting whether a patient will develop a disease, using a feature like "prescribed_medication_X" that is only given *after* the diagnosis. The model learns the correlation, but in production, new patients won't have this feature yet.

**Another example**: In fraud detection, using "account_flagged_for_review" as a feature—this flag may only exist *because* fraud was already detected.

Target leakage is particularly insidious because:
- The model may achieve near-perfect accuracy during training
- The leaked feature often has very high predictive power (because it's essentially encoding the answer)
- It only becomes apparent when the model fails in production

**Prevention**: Carefully audit each feature and ask: "Would this information be available at the moment of prediction in production?"

## Model selection leakage

This occurs when the test set is used to compare models, tune hyperparameters, or make any decision that influences the final model. Even though the test data is never directly used for training, the feedback from test performance guides the model selection process.

**Example**: Training several models (Linear Regression, Decision Tree, Random Forest), evaluating each on the test set, and choosing the one with the lowest test error. Then reporting that test error as the expected performance.

**The problem**: The reported test error is no longer an unbiased estimate of generalization performance. By selecting the model that performs best on *this specific* test set, we are implicitly fitting to the test data. This is often called **overfitting to the test set**.

### A worked example of what not to do

Let's make the mistake explicitly. We train two models with a minimal preprocessing pipeline and evaluate both on the test set.

In [ ]:
from sklearn.compose import make_column_transformer, make_column_selector
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

preprocessing = make_column_transformer(
    (make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), make_column_selector(dtype_include=np.number)),
    (OneHotEncoder(handle_unknown="ignore"), make_column_selector(dtype_include=object)),
)

lin_reg = make_pipeline(preprocessing, LinearRegression())
lin_reg.fit(X_train, y_train)
y_pred = lin_reg.predict(X_test)  # predictions on the TEST set

We can compare some of the predicted values with their actual labels, and see the percentage error of each prediction:

In [ ]:
print("Actual values:", y_test.iloc[:5].tolist())
print("Predictions:  ", y_pred[:5].round(-2).tolist())
error_ratios = y_pred[:5].round(-2) / y_test.iloc[:5] - 1
print("Errors:       ", ", ".join(f"{100 * ratio:.1f}%" for ratio in error_ratios))

Actual values: [47700.0, 45800.0, 500001.0, 218600.0, 278000.0]
Predictions:   [54300.0, 124400.0, 255700.0, 268200.0, 263000.0]
Errors:        13.8%, 171.6%, -48.9%, 22.7%, -5.4%


But the performance measure we established is the RMSE over the whole set:

In [ ]:
root_mean_squared_error(y_test, y_pred)

70059.19333925014

An error of about 70,000 dollars for predictions of house values with a median price of 180,000 dollars doesn't seem very useful. Let's try another model:

In [ ]:
from sklearn.tree import DecisionTreeRegressor

tree_reg = make_pipeline(preprocessing, DecisionTreeRegressor(random_state=42))
tree_reg.fit(X_train, y_train)
root_mean_squared_error(y_test, tree_reg.predict(X_test))

69175.76918851306

Whichever of the two is better, **we have just made the mistake**: by using the test set to compare models (Linear Regression vs. Decision Tree), we have committed model selection leakage. Even though we never trained on test data directly, the test set performance influenced our model choice. This means:

- The test error is now an **optimistically biased** estimate of true generalization performance
- We have effectively "overfit to the test set"
- The test set can no longer serve as an unbiased proxy for unseen data

**The solution**: Keep the test set in a "vault" and use it **only once**, at the very end. For comparing models and tuning hyperparameters, we need a separate **validation set**.

## Validation set

To avoid improperly using the test set during model development, the dataset is usually split into three parts:

- **Training**: To fit the model's **parameters**.
- **Validation**: To tune **hyperparameters** and **compare different models**. This set allows measuring intermediate performance without compromising the test set.
- **Test**: To evaluate the final model only once. It should remain untouched throughout the entire development process.

Alternatively, use **[cross-validation](cross_validation.ipynb)** on the training set to compare models without touching the test set at all. The test set should remain in a "vault" until the final evaluation.

## Summary: preventing data leakage

| Type | What leaks | Prevention |
|------|-----------|------------|
| **Feature leakage** | Test set statistics into preprocessing | Use pipelines; fit transformers only on training data |
| **Target leakage** | Future/derived information into features | Audit features for temporal/causal validity |
| **Model selection leakage** | Test performance into model choice | Use validation set or cross-validation; test only once |

The golden rule: **the test set must simulate truly unseen data**. Any information flow from test to training—whether through features, preprocessing, or feedback—violates this principle and invalidates the performance estimate.